### Table Creation

In [0]:
%sql
create schema silver;

In [0]:
%sql
drop table if exists silver.dim_customers;

create table silver.dim_customers(
    customer_key bigint generated always as identity,
    customer_id string,
    customer_unique_id string,
    customer_zip_code_prefix integer,
    customer_city string,
    customer_state string,
    dwh_date timestamp
);

alter table silver.dim_customers set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.dim_customers alter column dwh_date set default current_timestamp();

In [0]:
%sql
drop table if exists silver.dim_sellers;

create table silver.dim_sellers(
    seller_key bigint generated always as identity,
    seller_id string,
    seller_zip_code_prefix integer,
    seller_city string,
    seller_state string,
    dwh_date timestamp
);

alter table silver.dim_sellers set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.dim_sellers alter column dwh_date set default current_timestamp();

In [0]:
%sql
drop table if exists silver.dim_products;

create table silver.dim_products(
    product_key bigint generated always as identity,
    product_id string,
    product_category_name string,
    product_price double,
    product_name_length integer,
    product_description_length integer,
    product_photos_qty integer,
    product_weight_g integer,
    product_length_cm integer,
    product_height_cm integer,
    product_width_cm integer,
    dwh_date timestamp
);

alter table silver.dim_products set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.dim_products alter column dwh_date set default current_timestamp();

In [0]:
%sql
drop table if exists silver.dim_reviews;

create table silver.dim_reviews(
    review_key bigint generated always as identity,
    review_id string,
    review_score integer,
    review_comment_title string,
    review_comment_message string,
    review_creation_date timestamp,
    review_answer_timestamp timestamp,
    dwh_date timestamp
);

alter table silver.dim_reviews set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.dim_reviews alter column dwh_date set default current_timestamp();
alter table silver.dim_reviews alter column review_creation_date set default current_timestamp();

In [0]:
%sql
drop table if exists silver.dim_payments;

create table silver.dim_payments(
    payment_key bigint generated always as identity,
    order_id string,
    payment_sequential integer,
    payment_type string,
    payment_installments integer,
    payment_value double,
    dwh_date timestamp
);

alter table silver.dim_payments set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.dim_payments alter column dwh_date set default current_timestamp()

In [0]:
%sql
drop table if exists silver.fact_orders;

create table silver.fact_orders(
    sales_key bigint generated always as identity,
    customer_key bigint,
    product_key bigint,
    seller_key bigint,
    payment_key bigint,
    review_key bigint,
    order_id string,
    order_status string,
    order_purchase_timestamp timestamp,
    order_approved_at timestamp,
    order_delivered_carrier_date timestamp,
    order_delivered_customer_date timestamp,
    order_estimated_delivery_date timestamp,
    dwh_date timestamp
);

alter table silver.fact_orders set tblproperties ('delta.feature.allowColumnDefaults' = 'supported');

alter table silver.fact_orders alter column dwh_date set default current_timestamp();

### Insertion of Values

In [0]:
%sql
insert into silver.dim_customers (
    customer_id, 
    customer_unique_id, 
    customer_zip_code_prefix, 
    customer_city, 
    customer_state)
select 
    customer_id, 
    customer_unique_id, 
    customer_zip_code_prefix, 
    customer_city, 
    customer_state
from bronze.customers;

In [0]:
%sql
select * from silver.dim_customers limit 10;

In [0]:
%sql
insert into silver.dim_sellers (
    seller_id, 
    seller_zip_code_prefix, 
    seller_city, 
    seller_state)
select 
    seller_id, 
    seller_zip_code_prefix, 
    seller_city, 
    seller_state
from bronze.sellers;

In [0]:
%sql
select * from silver.dim_sellers limit 10;

In [0]:
%sql
insert into silver.dim_payments (
    order_id,
    payment_sequential, 
    payment_type, 
    payment_installments, 
    payment_value)
select 
    order_id,
    coalesce(payment_sequential, 0), 
    coalesce(payment_type, 'undefined'), 
    coalesce(payment_installments, 0), 
    coalesce(payment_value, 0.0)
from bronze.order_payments;

In [0]:
%sql
select * from silver.dim_payments limit 10;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
insert into silver.dim_reviews (
    review_id, 
    review_score, 
    review_comment_title, 
    review_comment_message, 
    review_creation_date, 
    review_answer_timestamp)
select 
    review_id, 
    coalesce(try_cast(review_score as int), 0) as review_score,
    review_comment_title, 
    review_comment_message, 
    try_cast(review_creation_date as timestamp) as review_creation_date,
    try_cast(review_answer_timestamp as timestamp) as review_answer_timestamp
from bronze.order_reviews r
where r.order_id in (
    select order_id from bronze.orders
    
) and try_cast(review_answer_timestamp as timestamp) is not null;

In [0]:
%sql
select 
    review_id, 
    coalesce(try_cast(review_score as int), 0) as review_score,
    review_comment_title, 
    review_comment_message, 
    try_cast(review_creation_date as timestamp) as review_creation_date,
    try_cast(review_answer_timestamp as timestamp) as review_answer_timestamp
from bronze.order_reviews r
where r.order_id in (
    select order_id from bronze.orders
) and try_cast(review_answer_timestamp as timestamp) is not null;

In [0]:
%sql
select distinct try_cast(review_score as int) from bronze.order_reviews

In [0]:
%sql
select distinct review_creation_date from bronze.order_reviews

In [0]:
%sql
select * from silver.dim_reviews limit 10;

In [0]:
%sql
insert into silver.dim_products(
    product_id, 
    product_category_name, 
    product_price, 
    product_name_length, 
    product_description_length, 
    product_photos_qty, 
    product_weight_g, 
    product_length_cm, 
    product_height_cm, 
    product_width_cm, 
    dwh_date)
select 
    p.product_id,
    p.product_category_name,
    coalesce(oi.price, 0) as product_price,
    coalesce(p.product_name_lenght, 0),
    coalesce(p.product_description_lenght, 0),
    coalesce(p.product_photos_qty, 0),
    coalesce(p.product_weight_g, 0),
    coalesce(p.product_length_cm, 0) as product_length_cm,
    coalesce(p.product_height_cm, 0) as product_height_cm,
    coalesce(p.product_width_cm, 0) as product_width_cm,
    current_timestamp()
from bronze.order_items oi
join bronze.orders o on oi.order_id = o.order_id
join bronze.products p on p.product_id = oi.product_id
where p.product_category_name is not null

In [0]:
%sql
select * from silver.dim_products limit 10

In [0]:
%sql
select 
    p.product_id,
    p.product_category_name,
    oi.price as product_price,
    p.product_name_lenght,
    p.product_description_lenght,
    p.product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,
    current_timestamp()
from bronze.order_items oi
join bronze.orders o on oi.order_id = o.order_id
join bronze.products p on p.product_id = oi.product_id

In [0]:
%sql
insert into silver.fact_orders (
    customer_key, 
    product_key, 
    seller_key, 
    payment_key, 
    review_key,
    order_id,
    order_status, 
    order_purchase_timestamp, 
    order_approved_at, 
    order_delivered_carrier_date, 
    order_delivered_customer_date, 
    order_estimated_delivery_date)
select 
    c.customer_key,
    p.product_key,
    s.seller_key,
    pay.payment_key,
    rr.review_key,
    o.order_id,
    o.order_status,
    try_cast(o.order_purchase_timestamp as timestamp) as order_purchase_timestamp,
    try_cast(o.order_approved_at as timestamp) as order_approved_at,
    try_cast(o.order_delivered_carrier_date as timestamp) as order_delivered_carrier_date,
    try_cast(o.order_delivered_customer_date as timestamp) as order_delivered_customer_date,
    try_cast(o.order_estimated_delivery_date as timestamp) as order_estimated_delivery_date
from bronze.orders o
left join silver.dim_customers c on o.customer_id = c.customer_id
left join bronze.order_items oi on oi.order_id = o.order_id
left join silver.dim_products p on p.product_id = oi.product_id
left join silver.dim_sellers s on s.seller_id = oi.seller_id
left join bronze.order_reviews  r on r.order_id = o.order_id
left join silver.dim_reviews rr on r.review_id = rr.review_id
left join silver.dim_payments pay on o.order_id = pay.order_id

In [0]:
%sql
select 
    o.order_id,
    c.customer_key,
    p.product_key,
    s.seller_key,
    pay.payment_key,
    rr.review_key,
    o.order_status,
    try_cast(o.order_purchase_timestamp as timestamp) as order_purchase_timestamp,
    try_cast(o.order_approved_at as timestamp) as order_approved_at,
    try_cast(o.order_delivered_carrier_date as timestamp) as order_delivered_carrier_date,
    try_cast(o.order_delivered_customer_date as timestamp) as order_delivered_customer_date,
    try_cast(o.order_estimated_delivery_date as timestamp) as order_estimated_delivery_date
from bronze.orders o
left join silver.dim_customers c on o.customer_id = c.customer_id
left join bronze.order_items oi on oi.order_id = o.order_id
left join silver.dim_products p on p.product_id = oi.product_id
left join silver.dim_sellers s on s.seller_id = oi.seller_id
left join bronze.order_reviews  r on r.order_id = o.order_id
left join silver.dim_reviews rr on r.review_id = rr.review_id
left join silver.dim_payments pay on o.order_id = pay.order_id

In [0]:
%sql
select * from silver.fact_orders limit 10